# Taller · Ajedrez gráfico y agentes inteligentes
## Del agente aleatorio a la búsqueda adversaria
**Inteligencia Artificial · Enunciado para estudiantes**

**Modalidad propuesta:** parejas. **Duración:** dos semanas desde la asignación. El docente comunica la fecha exacta. **Producto principal:** este cuaderno completado, ejecutado y acompañado de los módulos propios, partidas PGN y resultados CSV.

### Desafío
Construyan una aplicación gráfica de ajedrez que permita humano contra humano, humano contra agente y agente contra agente. Implementen varios agentes, expliquen sus decisiones y compárenlos con evidencia reproducible.

Este cuaderno entrega un visor gráfico y un controlador básico de jugadas humanas. Los agentes, la búsqueda, la evaluación y el torneo se desarrollan por ustedes. Los bloques de pseudocódigo especifican contratos y decisiones, no soluciones en Python.

**Propósitos:** aplicar agentes racionales, modelado de estados, búsqueda de capítulos 3–4 y métodos adversarios del capítulo 5 de AIMA; distinguir corrección de reglas, calidad de juego y eficiencia.


## 1. Requisitos de ejecución
Plataforma base: Jupyter Notebook o JupyterLab en computador. Colab puede requerir activar el gestor de widgets. No se exige compatibilidad con Pythonista.

Instalen manualmente en el entorno del kernel:

```bash
python -m pip install notebook python-chess ipywidgets matplotlib
```

`python-chess` se usa para reglas, movimientos legales, FEN, PGN y SVG. Los algoritmos de decisión deben ser propios. No se permiten Stockfish ni motores UCI como implementación de agentes. SimpleAI/AIMA-Python se pueden consultar y citar, pero el núcleo adversario del taller se implementa desde cero.

El tablero SVG es gráfico; la entrada base se realiza por selección de casillas en listas y botón. Deben ampliar esta base con selección de agentes y modos de partida. La interacción por clic directo en casillas es una mejora opcional.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Protocol
import importlib.util, random, time, csv
required = {"chess":"python-chess", "ipywidgets":"ipywidgets", "IPython":"ipython"}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
READY = not missing
if missing:
    print("Faltan dependencias:", ", ".join(missing))
    print("Instálelas manualmente y reinicie el kernel. Las tareas de diseño pueden completarse mientras tanto.")
else:
    import chess, chess.svg, chess.pgn
    import ipywidgets as widgets
    from IPython.display import display, SVG, clear_output
    print("Entorno listo. Versión del módulo chess:",chess.__version__)


## 2. Análisis antes de programar
Completen un PEAS para su sistema. Definan si medirán la fuerza por puntos, precisión táctica o rendimiento bajo tiempo limitado.

| Componente | Su especificación |
|---|---|
| Desempeño | [Completar: legalidad, puntos, tiempo, reproducibilidad] |
| Entorno | [Completar: tablero, reglas, rival y reloj] |
| Actuadores | [Completar] |
| Sensores | [Completar] |

Describan el estado con suficiente detalle para enroque, captura al paso y repeticiones. Clasifiquen el entorno y expliquen por qué un rival desconocido no vuelve estocásticas las reglas.

**Invariantes:** todo movimiento aplicado debe ser legal; las búsquedas no modifican la posición real; el turno cambia una sola vez por movimiento; una partida terminal no pide otra jugada a un agente.

**Política de tablas del taller:** reclamar automáticamente tablas cuando `outcome(claim_draw=True)` lo permita. Es una convención experimental explícita. El ajedrez real permite decidir si reclamar algunas tablas. El límite de plies del torneo produce un estado “truncada”, separado de las tablas reglamentarias.


## 3. Base gráfica suministrada
La clase siguiente permite mover piezas, escoger promoción, deshacer y reiniciar. Muestra turno, historial SAN y posibles destinos de la casilla elegida. El núcleo valida reglas con `python-chess`.

**Prueba manual:** e2–e4, e7–e5, g1–f3; intente después mover una pieza que no corresponde al turno. Compruebe que el estado no cambia. Para una promoción, cargue una posición específica en el tablero y use “Actualizar vista”.


In [ ]:
if READY:
    class ChessView:
        def __init__(self):
            self.board=chess.Board()
            self.origin=widgets.Dropdown(options=chess.SQUARE_NAMES,value="e2",description="Origen")
            self.target=widgets.Dropdown(options=chess.SQUARE_NAMES,value="e4",description="Destino")
            self.promotion=widgets.Dropdown(options=[("Dama","q"),("Torre","r"),("Alfil","b"),("Caballo","n")],description="Promoción")
            self.move_button=widgets.Button(description="Mover",button_style="primary")
            self.undo_button=widgets.Button(description="Deshacer")
            self.reset_button=widgets.Button(description="Nueva partida")
            self.refresh_button=widgets.Button(description="Actualizar vista")
            self.output=widgets.Output()
            self.status=widgets.Label()
            self.move_button.on_click(self.play)
            self.undo_button.on_click(self.undo)
            self.reset_button.on_click(self.reset)
            self.refresh_button.on_click(lambda _:self.render())
            self.origin.observe(lambda _:self.render(),names="value")
            self.ui=widgets.VBox([widgets.HBox([self.origin,self.target]),self.promotion,
                                  widgets.HBox([self.move_button,self.undo_button]),
                                  widgets.HBox([self.reset_button,self.refresh_button]),self.status,self.output])
            self.render()
        def render(self,message=""):
            outcome=self.board.outcome(claim_draw=True)
            turn="blancas" if self.board.turn else "negras"
            self.status.value=message or (f"Resultado: {outcome.result()} · {outcome.termination.name}" if outcome else f"Turno: {turn}")
            sq=chess.parse_square(self.origin.value)
            arrows=[(m.from_square,m.to_square) for m in self.board.legal_moves if m.from_square==sq]
            last=self.board.peek() if self.board.move_stack else None
            with self.output:
                clear_output(wait=True)
                display(SVG(chess.svg.board(self.board,size=460,lastmove=last,arrows=arrows,
                                           check=self.board.king(self.board.turn) if self.board.is_check() else None)))
                replay=self.board.root(); history=[]
                for m in self.board.move_stack:
                    history.append(replay.san(m)); replay.push(m)
                print("Historial SAN:"," ".join(history) or "Sin jugadas")
        def play(self,_=None):
            if self.board.outcome(claim_draw=True):
                self.render("Partida terminada según la política de tablas del taller"); return
            src=chess.parse_square(self.origin.value); dst=chess.parse_square(self.target.value)
            piece=self.board.piece_at(src)
            promo=None
            if piece and piece.piece_type==chess.PAWN and chess.square_rank(dst) in (0,7):
                promo={"q":chess.QUEEN,"r":chess.ROOK,"b":chess.BISHOP,"n":chess.KNIGHT}[self.promotion.value]
            move=chess.Move(src,dst,promotion=promo)
            if move not in self.board.legal_moves:
                self.render("Movimiento ilegal. El tablero conserva su estado."); return
            self.board.push(move); self.render()
        def undo(self,_=None):
            if self.board.move_stack: self.board.pop()
            self.render()
        def reset(self,_=None):
            self.board=chess.Board(); self.render()
    view=ChessView()
    display(view.ui)


### Ampliaciones gráficas obligatorias
1. Selector independiente de agente para blancas y negras, incluyendo humano.
2. Botones “Jugar turno del agente”, “Iniciar/Pausar” y “Nueva partida”.
3. Profundidad, presupuesto temporal y semilla configurables; registro visible de la decisión.
4. Estado de la partida, jaque, último movimiento y resultados de búsqueda por turno.
5. Exportación de partida PGN y tabla CSV de métricas.

Mantengan la interfaz utilizable durante una búsqueda: un callback que ejecuta un torneo completo de forma síncrona bloqueará el kernel y el botón Pausar. Pueden empezar con un turno por botón y luego implementar una tarea cooperativa con cesiones periódicas y cancelación. El agente trabaja sobre una copia y el controlador aplica la jugada solo si sigue correspondiendo al estado vigente.

El botón Deshacer debe tener una política explícita: un ply o una pareja de turnos en modo humano contra agente. No alteren el estado mientras una búsqueda está activa sin cancelarla primero.


## 4. Arquitectura y contrato común
Separen reglas, decisión, controlador, vista y experimentos. Usen clases y `dataclasses` para configuración y métricas.

| Componente | Responsabilidad |
|---|---|
| `ChessEnvironment` | Estado real, validación y aplicación de movimientos |
| `Agent` | Elegir una jugada sobre una copia, sin tocar la vista |
| `Evaluator` | Evaluación consistente desde blancas o desde una perspectiva documentada |
| `SearchStats` | Nodos, hojas, podas, profundidad, tiempo y simulaciones |
| `GameController` | Alternancia, presupuesto, cancelación y terminación |
| `ChessView` | Controles, tablero y mensajes |
| `Tournament` | Emparejamientos, semillas, CSV y PGN |

El siguiente contrato está suministrado; no implementa agentes. `None` se devuelve únicamente cuando el estado ya es terminal según la política acordada. Un vencimiento de plazo debe devolver la mejor jugada legal disponible, no `None`.


In [ ]:
@dataclass
class SearchStats:
    nodes: int = 0
    leaves: int = 0
    cutoffs: int = 0
    completed_depth: int = 0
    elapsed_s: float = 0.0
    simulations: int = 0

@dataclass
class Decision:
    move: object | None
    score: float | None
    stats: SearchStats

class Agent(Protocol):
    name: str
    def choose(self, board, *, time_limit_s: float, seed: int) -> Decision:
        """Recibe copia con historia. Devuelve jugada legal y métricas."""
        ...

# TODO: escriban sus clases en las celdas de implementación siguientes.


## 5. Agentes que deben construir
Se requieren **seis agentes**. Todos comparten el contrato y la política de terminación.

| Agente | Decisión | Qué permite estudiar |
|---|---|---|
| Aleatorio | Elegir uniformemente entre movimientos legales | Línea base y semillas |
| Voraz de un ply | Evaluar cada sucesor inmediato | Sesgo de corto plazo |
| Minimax | Alternar MAX/MIN hasta profundidad fija | Respuesta adversaria |
| Alfa-beta | Mismo árbol y evaluación, con poda | Ahorro sin cambiar el valor de raíz |
| Alfa-beta mejorado | Ordenación + profundización iterativa con plazo | Calidad bajo recursos limitados |
| MCTS-UCT | Selección, expansión, rollout y respaldo | Exploración frente a explotación |

En el último agente documenten una política de rollout y un límite. Un rollout truncado puede recibir una evaluación normalizada a la misma escala de recompensas, pero debe registrarse como truncado, no como tabla. Elijan la jugada final por una regla documentada, por ejemplo mayor número de visitas.

**Mejora adicional a escoger:** quietud; tabla de transposición con tipos de cota y profundidad; o evaluación cuyos pesos ajustan mediante búsqueda local. Al menos una debe implementarse y compararse mediante ablación.

### Evaluación común
Propongan $E(s)=w_mM(s)+w_pP(s)+w_kK(s)+w_aA(s)$, o una alternativa justificada. Definan cada característica, escala, signo y costo. Los valores terminales se resuelven antes de consultar $E$ y deben dominar a las estimaciones no terminales. Si bonifican mates más cortos, documenten el ajuste por distancia y su interacción con la caché.

Por ejemplo, pueden medir material y movilidad, añadir estructura de peones y seguridad del rey. No basta con asignar números a las piezas: expliquen qué decisiones pretende inducir cada término. La evaluación de negras debe favorecer a negras cuando ese color decide, de acuerdo con la perspectiva que hayan elegido.


### Pseudocódigo orientador
```text
DECIDIR_MINIMAX(estado, profundidad):
    si estado es terminal: devolver su utilidad exacta
    si profundidad es cero: devolver E(estado)
    obtener acciones legales
    evaluar sucesores alternando MAX y MIN
    respaldar el máximo o mínimo según el turno
    devolver valor y primera acción de una variante óptima

DECIDIR_ALFABETA(estado, profundidad, alfa, beta):
    comprobar terminal antes de corte
    recorrer los sucesores en el orden elegido
    actualizar el mejor valor y la cota correspondiente
    detener el recorrido cuando las cotas justifican la poda
    registrar el evento de poda y devolver valor y acción

PROFUNDIZACION_ITERATIVA(estado, plazo):
    guardar una jugada legal de respaldo
    repetir con profundidades crecientes
        buscar sin exceder el plazo
        si la iteración termina: reemplazar la decisión guardada
        si no termina: conservar la última decisión completa
    devolver decisión y profundidad realmente completada

MCTS(estado, presupuesto):
    crear raíz con historial y jugador de referencia
    repetir hasta agotar presupuesto
        seleccionar usando UCT con perspectiva correcta
        expandir una acción todavía no explorada
        simular hasta terminal o límite documentado
        retropropagar visitas y recompensa de forma consistente
    escoger un hijo con la regla final definida
```

**Decisiones que deben resolver:** dónde verificar el plazo, qué cuenta como nodo, cómo desempatar, cómo restaurar un tablero tras una excepción y qué perspectiva conserva cada valor. Usar `push` debe acompañarse de `pop` en un bloque `finally` cuando corresponda.


In [ ]:
# IMPLEMENTACIÓN A: agente aleatorio, agente voraz y evaluación.
# Complete con su código. Verifique reproducibilidad y perspectiva de color.


In [ ]:
# IMPLEMENTACIÓN B: minimax y alfa-beta, con las mismas hojas y evaluación.
# Complete con su código. Primero compare a profundidad 1 y 2.


In [ ]:
# IMPLEMENTACIÓN C: ordenación y profundización iterativa con presupuesto.
# Complete con su código. Preserve la última iteración terminada.


In [ ]:
# IMPLEMENTACIÓN D: MCTS-UCT y mejora adicional elegida.
# Complete con su código. Documente recompensas y simulaciones truncadas.


## 6. Integración del controlador
```text
AL_SOLICITAR_TURNO:
    si hay búsqueda activa o partida terminada: informar y salir
    identificar agente según el color al turno
    si es humano: esperar una jugada por la interfaz
    si es agente:
        capturar identificador de versión del estado real
        copiar el tablero conservando la historia
        solicitar Decision al agente
        comprobar plazo, legalidad y que la versión siga vigente
        aplicar exactamente una jugada al tablero real
        registrar métricas y actualizar vista
```

El historial debe sobrevivir a las copias: `board.copy(stack=True)` conserva la pila. Una clave de búsqueda basada solo en la colocación de piezas no detecta por sí misma todas las condiciones de tablas.

Si adoptan un límite temporal estricto de torneo, el controlador debe poder detectar su incumplimiento. Una medición después de devolver no interrumpe un agente bloqueado: documenten la diferencia entre verificación cooperativa y aislamiento en proceso. Para el alcance mínimo basta presupuesto cooperativo y registro del exceso.


In [ ]:
# IMPLEMENTACIÓN E: controlador, selector de agentes y actualización de la vista.
# Reutilice ChessView o adapte su arquitectura conservando la validación legal.


## 7. Todo lo visto: actividades de integración
Estas actividades son obligatorias, además de los seis agentes. No se exige forzar BFS, UCS o A* como jugadores completos.

### A. Búsqueda no informada en un subproblema cooperativo
Construyan un microproblema con una torre en un tablero vacío, sin reyes ni adversario: origen a1, destino h8, acciones de torre y costo unitario por movimiento. Resuelvan con BFS y DFS, y con UCS asignando a cada movimiento el número de casillas recorridas. Definan estados, acciones y costos; distingan longitud de ruta y costo.

Analicen por qué encontrar una secuencia a un objetivo bajo control de todos los movimientos no garantiza mate contra un rival. La actividad usa reglas simplificadas explícitas, no posiciones legales completas de ajedrez.

### B. Búsqueda informada y ordenación
Resuelvan el microproblema anterior con A* y voraz. Para costo unitario pueden usar $h=0$ en destino, $h=1$ si comparte fila o columna y $h=2$ en otro caso. Para costo por distancia estudien la distancia Manhattan. Justifiquen admisibilidad para A* respecto al costo adoptado y comparen expansiones. Después expliquen la analogía y las diferencias con ordenar jugadas para alfa-beta: ordenar no altera por sí mismo el conjunto de respuestas rivales.

### C. Búsqueda local para ajustar la evaluación
Definan un vector de pesos y una función objetivo sobre posiciones de entrenamiento. Realicen una búsqueda local pequeña de pesos con presupuesto explícito. Separen entrenamiento y prueba; no elijan pesos usando resultados de prueba. Reporten pesos iniciales/finales y una ablación. Si esta es su mejora adicional, la implementación cuenta para ambos apartados.

### D. Incertidumbre y garantías
1. Representen un problema táctico de mate forzado como OR para el atacante y AND para el defensor. Expliquen por qué deben cubrir todas las defensas.
2. En un árbol pequeño con azar, calculen expectiminimax y contrasten con minimax. Definan probabilidades y escala de utilidad.
3. Describan cómo cambiarían percepción y estado interno para Kriegspiel. Den un ejemplo de información que su agente de ajedrez actual usa y que allí no podría consultar.

**No cambien las reglas del ajedrez clásico para introducir dados.** El ejemplo estocástico se realiza por separado.


In [ ]:
# INTEGRACIÓN: microproblema de torre, búsquedas y ajuste local de pesos.
# Agregue tantas celdas como necesite e incluya gráficas de las comparaciones.


## 8. Pruebas de corrección
El paquete de posiciones siguiente es un punto de partida. Las expectativas se deben verificar mediante reglas y análisis, sin asumir que siempre exista una única mejor jugada. Añadan al menos dos posiciones propias con justificación.

| Caso | Criterio |
|---|---|
| Inicial | 20 movimientos legales |
| Enroque | Ambos enroques blancos disponibles en la posición dada |
| Promoción | Cuatro promociones legales de a7 a a8 |
| Captura al paso | e5d6 legal y captura al paso |
| Mate en uno | El agente debe elegir alguna jugada que deje mate |
| Mate terminal | Sin decisión nueva, utilidad de derrota para negras |
| Ahogado | Tablas sin estar en jaque |
| Repetición | Reclamo detectable desde una historia construida |

A igual profundidad y evaluación, minimax y alfa-beta deben coincidir en valor. Las jugadas pueden diferir por empate. Una búsqueda debe dejar intactos FEN y pila de movimientos del tablero recibido. Si se interrumpe, ese invariante sigue aplicando.


In [ ]:
POSITIONS = {
    "initial": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "castling": "r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1",
    "promotion": "7k/P7/8/8/8/8/8/7K w - - 0 1",
    "en_passant": "7k/8/8/3pP3/8/8/8/7K w - d6 0 2",
    "mate_in_one": "7k/8/5KQ1/8/8/8/8/8 w - - 0 1",
    "checkmate": "7k/6Q1/5K2/8/8/8/8/8 b - - 0 1",
    "stalemate": "7k/5K2/6Q1/8/8/8/8/8 b - - 0 1",
}
if READY:
    for name,fen in POSITIONS.items():
        b=chess.Board(fen)
        assert b.is_valid(), (name,b.status())
        print(name,"movimientos legales:",b.legal_moves.count())
    assert chess.Board(POSITIONS["initial"]).legal_moves.count()==20
    assert chess.Move.from_uci("e1g1") in chess.Board(POSITIONS["castling"]).legal_moves
    assert chess.Move.from_uci("e1c1") in chess.Board(POSITIONS["castling"]).legal_moves
    assert chess.Board(POSITIONS["checkmate"]).is_checkmate()
    assert chess.Board(POSITIONS["stalemate"]).is_stalemate()
    repetition=chess.Board()
    for uci in ["g1f3","g8f6","f3g1","f6g8"]*2:
        repetition.push_uci(uci)
    assert repetition.can_claim_threefold_repetition()
    print("Reglas de las posiciones base verificadas.")


initial movimientos legales: 20
castling movimientos legales: 26
promotion movimientos legales: 7
en_passant movimientos legales: 5
mate_in_one movimientos legales: 23
checkmate movimientos legales: 0
stalemate movimientos legales: 0
Reglas de las posiciones base verificadas.


In [ ]:
# PRUEBAS DE SUS AGENTES:
# 1. Minimax frente a alfa-beta a igual profundidad/evaluación.
# 2. Decisiones legales de todos los agentes en posiciones no terminales.
# 3. Victoria inmediata y evasión de jaque.
# 4. Tablero e historia intactos tras búsqueda normal e interrumpida.
# 5. Misma semilla/presupuesto por simulaciones produce resultado reproducible.
# 6. Profundización devuelve jugada legal al agotarse el presupuesto.
# 7. MCTS elige para el rival sin maximizar accidentalmente al jugador raíz.
# Implemente y muestre los resultados, incluidos fallos que haya corregido.


## 9. Protocolo experimental y torneo
### Experimento 1: ahorro de búsqueda
Usen cinco posiciones intermedias válidas con distinto carácter: captura, jaque, defensa, posición tranquila y final. Comparen minimax, alfa-beta natural y alfa-beta ordenado a profundidades 1, 2 y 3 con evaluación fija. Mantengan el mismo árbol y condiciones de corte. Muestren valores y métricas; separen el tiempo de ordenar del tiempo total si lo instrumentan.

### Experimento 2: fuerza de juego bajo presupuesto
Torneo de los seis agentes, todos contra todos. Cada par juega al menos **cuatro partidas**, dos con cada color, desde posiciones iniciales/aperturas equilibradas y con semillas distintas. Son 60 partidas como mínimo. Compartan las mismas aperturas entre emparejamientos e intercambien colores. Usen, por ejemplo, un presupuesto de 0.2 segundos por decisión ajustado a su hardware y un límite de 160 plies. Si un algoritmo de profundidad fija excede el presupuesto, registren el exceso y apliquen una política uniforme documentada.

Puntuación reglamentaria: victoria 1, tablas 0.5 y derrota 0. Un límite artificial produce “truncada”: reporten su proporción por separado y dejen su resultado oficial sin asignar. Pueden presentar una clasificación provisional de partidas terminadas con la cobertura de cada agente; no den tablas ni victorias por evaluación estática como si fueran resultados oficiales.

### Experimento 3: ablación
Comparen una mejora con su versión base manteniendo los demás factores fijos. Por ejemplo, alfa-beta con/sin ordenación o evaluación con pesos iniciales/ajustados. Si comparan quietud con ausencia de quietud a igual profundidad, aclaren que el árbol ya no es el mismo.

### Experimento 4: MCTS
Repitan un conjunto de tácticas con al menos tres presupuestos de simulación y cinco semillas. Reporten tasa de acierto táctico, media y dispersión temporal. Una curva de mejora promedio no implica mejora monótona por semilla.

**Reporte mínimo:** nodos, hojas, podas, profundidad completada, tiempo por jugada, simulaciones, resultados, excesos y truncamientos. Para comparar fuerza use presupuesto temporal; para estudiar alfa-beta use profundidad fija. Para reproducibilidad exacta de MCTS use simulaciones fijas, porque un plazo temporal depende de la máquina.


In [ ]:
# Esquema orientador de CSV. Escriba archivos solo cuando ejecute sus experimentos.
CSV_FIELDS = ["game_id","opening_id","seed","white_agent","black_agent","ply",
              "move_uci","nodes","leaves","cutoffs","completed_depth",
              "elapsed_s","simulations","time_exceeded","termination","official_result"]
# TODO: torneo, exportación y gráficos. Preserve configuración y versiones usadas.


### Tablas para completar
| Agente | Puntos / partidas terminadas | Victorias | Tablas | Derrotas | Truncadas | Tiempo medio |
|---|---|---|---|---|---|---|
| Aleatorio | | | | | | |
| Voraz | | | | | | |
| Minimax | | | | | | |
| Alfa-beta | | | | | | |
| Alfa-beta mejorado | | | | | | |
| MCTS | | | | | | |

Incluyan barras de nodos por algoritmo y posición, curvas de profundidad contra tiempo y tasa de éxito de MCTS por presupuesto. No comparen números de “nodos” de implementaciones con definiciones distintas sin explicarlo.


## 10. Cronograma propuesto
| Días | Hito verificable |
|---|---|
| 1–2 | PEAS, contratos, reglas y tablero gráfico funcionando |
| 3–4 | Aleatorio, voraz, evaluación y pruebas de perspectiva |
| 5–6 | Minimax y alfa-beta con equivalencia de valor |
| 7–8 | Iterative deepening, plazo e integración de modos |
| 9–10 | MCTS y mejora adicional |
| 11–12 | Integraciones de capítulos 3–4 y experimentos |
| 13–14 | Análisis, limpieza del cuaderno y sustentación |

No aumenten la profundidad hasta verificar legalidad, restauración del estado y tratamiento de terminales. Diseñen el torneo primero con dos agentes y una partida corta.


## 11. Rúbrica · 100 puntos
| Criterio | Puntos | Evidencia |
|---|---:|---|
| Formulación, PEAS y arquitectura | 10 | Modelo, contratos y justificación |
| Interfaz y reglas | 15 | Tres modos, controles, turnos y pruebas especiales |
| Seis agentes | 30 | Aleatorio 2, voraz 3, minimax 6, alfa-beta 6, mejorado 6, MCTS 7 |
| Integración de capítulos 3–4 | 10 | Torre, búsqueda informada/local e incertidumbre |
| Mejora adicional y ablación | 10 | Comparación controlada y explicación |
| Experimentos reproducibles | 15 | Torneo, métricas, semillas, PGN y CSV |
| Informe y sustentación individual | 10 | Interpretación y modificación de código en vivo |
| **Total** | **100** | |

Conversión sugerida a escala de 0 a 5: puntos/20, sujeta a las reglas de evaluación del docente. La existencia de una interfaz vistosa no reemplaza la implementación de búsqueda. Los resultados inventados no cuentan como experimentos.

### Uso de herramientas y autoría
Declaren librerías, fuentes y ayudas de IA utilizadas. Cada integrante debe explicar el código, ejecutar una prueba nueva y modificar una decisión del algoritmo durante la sustentación. No se exige ocultar las herramientas usadas; se exige comprender y verificar lo entregado.


## 12. Sustentación y conclusiones
Respondan con evidencia de su implementación:
1. Muestre una posición donde el voraz falla y minimax encuentra una defensa.
2. Trace a mano una poda real de su registro. ¿Por qué es válida?
3. ¿Por qué minimax y alfa-beta podrían elegir jugadas distintas con el mismo valor?
4. Cambie la perspectiva de evaluación. ¿Qué errores aparecerían si solo cambia un signo?
5. ¿Qué retorna su programa si vence el plazo durante una iteración?
6. ¿Cómo evita que MCTS elija respuestas favorables al adversario equivocado?
7. ¿Qué conserva el historial que no contiene un FEN aislado?
8. ¿Qué parte del microproblema de torre deja de servir cuando existe oposición?
9. ¿Cómo sabe que los pesos ajustados no se sobreajustaron?
10. ¿Qué evidencia tiene para afirmar que un agente es mejor y bajo qué presupuesto?

**Conclusión individual:** describa una decisión de diseño que cambió después de observar un fallo. Incluya el caso que lo reveló y cómo comprobó la corrección.

## Lista de entrega
- Cuaderno ejecutado, con integrantes y conclusiones individuales.
- Módulos propios y README con instalación y ejecución.
- Dependencias con versiones del entorno probado.
- CSV de decisiones y resultados, PGN y configuración del torneo.
- Posiciones de entrenamiento/prueba separadas y análisis de ablación.
- Declaración de fuentes y ayudas utilizadas.

Los datos de salida deben provenir de sus ejecuciones. Los espacios vacíos de este enunciado son parte del trabajo por realizar.

### Referencias
- Russell y Norvig, *AIMA*, 4.ª edición, capítulos 2–5. [Índice oficial](https://aima.cs.berkeley.edu/contents.html).
- [python-chess: reglas y estados](https://python-chess.readthedocs.io/en/latest/core.html).
- [python-chess: SVG](https://python-chess.readthedocs.io/en/latest/svg.html).
- [python-chess: PGN](https://python-chess.readthedocs.io/en/latest/pgn.html).
- [ipywidgets: documentación](https://ipywidgets.readthedocs.io/en/stable/).
